In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

origem = '/content/drive/MyDrive/datasus/cnvtocsv'

destino = '/content/drive/MyDrive/datasus/cnvtocsv'

os.makedirs(destino, exist_ok=True)

Mounted at /content/drive


In [3]:
import csv
import re
import zipfile
from pathlib import Path

os.makedirs(destino, exist_ok=True)


def ler_cnv(caminho):

    nome = os.path.basename(caminho).upper()

    with open(caminho, "r", encoding="cp1252") as arquivo:
        linhas = arquivo.read().splitlines()

    registros = []

    for numero_linha, linha in enumerate(linhas[1:], start=2):

        if not linha.strip():
            continue

        if nome.startswith("CID10"):

            m = re.match(r"^\s*(\d+)\s{2}(.*)$", linha)

            if not m:
                raise ValueError(f"{nome}, linha {numero_linha}: registro inválido")

            ordem, restante = m.groups()
            partes = restante.rstrip().split(None, 1)

            if len(partes) != 2:
                raise ValueError(f"{nome}, linha {numero_linha}: registro CID10 inválido")

            codigo, descricao_com_cauda = partes
            descricao = descricao_com_cauda[:-5].strip()

            if not descricao:
                raise ValueError(f"{nome}, linha {numero_linha}: descrição vazia")

        elif nome.startswith("SEXO"):

            m = re.match(r"^\s*(\d+)\s{2}", linha)

            if not m or len(linha) < 61:
                raise ValueError(f"{nome}, linha {numero_linha}: registro SEXO inválido")

            ordem = m.group(1)
            descricao = linha[m.end():60].strip()
            codigo = linha[60:].strip().lstrip(",").strip()

            if not descricao or not codigo:
                raise ValueError(f"{nome}, linha {numero_linha}: descrição/código vazio")

        else:

            m = re.match(
                r"^\s*(\d+)\s{2}(.*?)\s{2,}(\S.*?)\s*$",
                linha
            )

            if not m:
                raise ValueError(f"{nome}, linha {numero_linha}: registro inválido")

            ordem, descricao, codigo = m.groups()

            descricao = descricao.strip()
            codigo = codigo.strip()

        registros.append([ordem, descricao, codigo])

    m = re.match(r"^(?:N\s+)?(\d+)", linhas[0])

    if not m:
        raise ValueError(f"Cabeçalho inválido em {nome}")

    quantidade_declarada = int(m.group(1))

    if len(registros) != quantidade_declarada:
        raise ValueError(
            f"{nome}: declarados {quantidade_declarada}, "
            f"lidos {len(registros)}"
        )

    return registros


def converter_cnv_para_csv(caminho_cnv, pasta_saida):

    registros = ler_cnv(caminho_cnv)

    nome_base = os.path.splitext(
        os.path.basename(caminho_cnv)
    )[0]

    caminho_csv = os.path.join(
        pasta_saida,
        nome_base + ".csv"
    )

    with open(
        caminho_csv,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as arquivo:

        escritor = csv.writer(arquivo, delimiter=";")

        escritor.writerow(["ordem", "descricao", "codigo"])
        escritor.writerows(registros)

    return caminho_csv, len(registros)


arquivos_cnv = sorted(
    [
        arquivo
        for arquivo in Path(origem).iterdir()
        if arquivo.is_file()
        and arquivo.suffix.upper() == ".CNV"
    ]
)

if not arquivos_cnv:
    raise FileNotFoundError("Nenhum arquivo CNV encontrado.")

resultados = []
csvs_gerados = []

for arquivo in arquivos_cnv:

    caminho_csv, quantidade = converter_cnv_para_csv(
        str(arquivo),
        destino
    )

    csvs_gerados.append(caminho_csv)

    resultados.append({
        "arquivo": arquivo.name,
        "registros": quantidade,
        "status": "CONVERTIDO"
    })


caminho_zip = os.path.join(
    destino,
    "arquivos_CNV_convertidos.zip"
)

with zipfile.ZipFile(
    caminho_zip,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for caminho_csv in csvs_gerados:
        zipf.write(
            caminho_csv,
            arcname=os.path.basename(caminho_csv)
        )
